**TF-IDF算法流程：**

**（1）对原文本进行分词，做词性标注和去除停用词等数据预处理操作，得到候选关键词。**

**（2）计算某词语在原文本中的词频 TF。**

**（3）计算该词语在整个语料库的逆文档频率 IDF。**

**（4）计算该词语的 TF-IDF 值（TF*IDF），并重复（2）～（4）步骤，得到所有候选关键词的 TF-IDF 值。**
        
**（5）对候选关键词的 TF-IDF 值进行倒序排列，得到排名前 Top-N 个词汇作为文本关键词。**

In [1]:
"""
导入模块
"""
import pandas as pd
import numpy as np
import jieba.posseg
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
#读取数据集（语料）
dataFile = 'data/data176649/sample_data - Copy.csv'
data = pd.read_csv(dataFile)
# 停用词表
stopWord = 'work/stopWord.txt'
#停用词列表
stopkey = [w.strip() for w in open(stopWord, 'r',encoding="UTF-8").readlines()]
# print(f"stopkey的内容是{stopkey}")

In [3]:
# 数据预处理操作：分词，去停用词，词性筛选
#text：待提取的文本
#stopkey：停用词列表
def dataPrepos(text,stopkey):
    l = []
    #规定关键词的词性必须属于pos
    pos = ['n','nz','v', 'vd', 'vn', 'l', 'a', 'd'] # 定义选取的词性

    #1.利用jieba将语句分词--->seg为分词之后得到的列表
    #这样分词后会得到词以及词性
    #词性标注：jieba.posseg.cut(), 返回一个迭代器
    seg = jieba.posseg.cut(text) #分词
    for i in seg:
        if i.word not in stopkey and i.flag in pos:# 去停用词 + 词性筛选
            l.append(i.word)
    #l为满足条件的关键词的候选列表
    return l

In [4]:
def getKeywords_tfidf(data, stopkey, topk):
    idList, titleList, abstractList = data['id'], data['title'], data['abstract']
    # print(type(titleList))    <class 'pandas.core.series.Series'>
    #Series类型由一组数据及与之相关的数据索引组成。简单来说，Series是一维带“标签”数组
    corpus = []  # 将所有文档输出到一个list中，一行就是一个文档
    for index in range(len(idList)):
        # 拼接标题和摘要 text为拼接之后的内容：标题。摘要
        text = '%s。%s' % (titleList[index], abstractList[index])

        # 文本预处理--->处理之后的数据为去掉了停用词以及不满足词性的候选词列表
        text = dataPrepos(text, stopkey)
        
        text = " ".join(text)  # 连接成字符串，空格分隔
        corpus.append(text)
    #最终corpus里面的内容有个10个元素（因为一共有10篇文章），每一个元素为数据处理之后的候选词。
    # print(f"corpus的内容是{corpus}")


    # 1、构建词频矩阵
    #初始化一个统计词频对象CountVectorizer函数用来构建语料库的中的词频矩阵
    # CountVectorizer是通过fit_transform函数将文本中的词语转换为词频矩阵，矩阵元素weight[i][j]
    # 表示j词在第i个文本下的词频，即各个词语出现的次数
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(corpus)  # 词频矩阵,a[i][j]:表示j词在第i个文本中的词频

    # 2、统计每个词的TF-IDF权值
    #TfidfTransformer是统计CountVectorizer中每个词语的tf-idf权值
    #TfidfTransformer也有个fit_transform函数，它的作用是计算tf - idf值。
    transformer = TfidfTransformer()
    tfidf = transformer.fit_transform(X)

    # 3、获取词袋模型中的关键词
    #获取词袋模型中的所有词语
    # 新版 sklearn 使用 get_feature_names_out，旧版使用 get_feature_names。
    if hasattr(vectorizer, "get_feature_names_out"):
        word = vectorizer.get_feature_names_out()
    else:
        word = vectorizer.get_feature_names()

    # 4、获取TF-IDF矩阵，a[i][j]表示j词在i篇文本中的TF-IDF权重
    weight = tfidf.toarray()

    # 5、打印词语权重
    ids, titles, keys = [], [], []
    for i in range(len(weight)):
        # print(u"-------这里输出第", i + 1, u"篇文本的词语tf-idf------")
        ids.append(idList[i])
        titles.append(titleList[i])
        df_word, df_weight = [], []  # 当前文章的所有词汇列表、词汇对应权重列表
        for j in range(len(word)):
            # print(word[j], weight[i][j])
            df_word.append(word[j])
            df_weight.append(weight[i][j])

        df_word = pd.DataFrame(df_word, columns=['word'])
        df_weight = pd.DataFrame(df_weight, columns=['weight'])
        #word_weight为224行 × 2列   224：词袋模型中的所有词语的个数
        word_weight = pd.concat([df_word, df_weight], axis=1)  # 拼接词汇列表和权重列表
        #print(word_weight)
        word_weight = word_weight.sort_values(by='weight', ascending=False)  # 按照权重值降序排列

        keyword = np.array(word_weight['word'])  # 选择词汇列并转成数组格式
        #print(keyword)
        #print(type(keyword))  <class 'numpy.ndarray'>
        word_split = [keyword[x] for x in range(0, topk)]  # 抽取前topK个词汇作为关键词
        word_split = " ".join(word_split)
        keys.append(word_split)
    result = pd.DataFrame({"id": ids, "title": titles, "key": keys}, columns=['id', 'title', 'key'])
    return result

In [5]:
result = getKeywords_tfidf(data, stopkey, 10)
print(f"result的内容是{result}")
# result.to_csv("work/keys_TFIDF.csv", index=False)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\1\AppData\Local\Temp\jieba.cache
Loading model cost 0.527 seconds.
Prefix dict has been built successfully.


result的内容是   id                    title                               key
0   1    永磁电机驱动的纯电动大巴车坡道起步防溜策略     电机 永磁 防溜 控制 策略 单元 踏板 力矩 转速 整车
1   2                机动车辆车门的肘靠   溃缩 机动车辆 车门 结构 事件 交叉 乘车 元件 峰值 衬板
2   3                  仪表板支撑结构    支架 支撑 横向 仪表板 端部 偏压 导引 接到 车身 邻接
3   4                  铰接的头枕总成   构件 头枕 座椅 靠背 联接 支承 枢转地 角度 驱动器 总成
4   5        用于评估和控制电池系统的系统和方法     电池 系统 分部 总和 容量 计算 控制 利用 数值 评估
5   6                    侧气囊装置    织物 结合部 结构 内管 膨胀 分隔 主体 气囊 横向 装置
6   7                  制造气囊的方法     织物 主体 步骤 结构 部上 制造 安装 重叠 气囊 部分
7   8                 上部椅背枢轴系统     椅背 枢轴 悬挂 组件 位置 耦接 结构 支承 竖直 枢转
8   9  用于在机动车的行驶过程中关闭和启动内燃机的方法  内燃机 机动车 方法 关闭 启动 行驶 手动 停转 过程 变速器
9  10                半倾斜货箱卸载系统  倾斜 货箱 纵向 部分 最靠近 运输工具 卸载 具有 布置 降低


**work文件夹中的news.csv包括新浪新闻8个领域（体育，娱乐，彩票，房产，教育，游戏，科技，股票）约800条新闻文本，字段包括content，type。**

**请面向后续的关键词提取任务，设计文本预处理方案（如自定义停用词表等文件，上传文件必须为UTF-8编码的csv文件），务必加详细注释，提高代码可读性**

In [6]:
# 读取新闻、停用词，并把同一类别的新闻合并为一篇“类别文档”。
# 从 02 目录打开笔记本时用 work；从项目根目录运行时用 02/work。
from pathlib import Path
import jieba
import jieba.analyse
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

work_dir = Path("work") if Path("work/news.csv").exists() else Path("02/work")
news = pd.read_csv(work_dir / "news.csv", encoding="utf-8")
stopwords = set((work_dir / "stopWord.txt").read_text(encoding="utf-8").splitlines())
# 对两个 jieba 关键词提取器也应用同一份停用词表。
jieba.analyse.set_stop_words(str(work_dir / "stopWord.txt"))

def clean_words(content):
    """分词后去掉停用词、空白、单字和纯数字，返回候选词列表。"""
    words = jieba.lcut(str(content))
    return [word.strip() for word in words
            if len(word.strip()) > 1
            and word.strip() not in stopwords
            and not word.strip().isdigit()]

# groupby(sort=False) 保留 CSV 中类别首次出现的顺序。
# 每个类别的全部新闻合并后，三种方法都使用同一份预处理文本。
category_texts = {}
for news_type, group in news.groupby("type", sort=False):
    all_words = []
    for content in group["content"].fillna(""):
        all_words.extend(clean_words(content))
    category_texts[news_type] = " ".join(all_words)

print(f"共读取 {len(news)} 条新闻，包含 {len(category_texts)} 个类别。")


共读取 799 条新闻，包含 8 个类别。


**使用TF-IDF算法提取news.csv中8个领域新闻的前20个关键词，将type,tfidfkey保存到"work/学号.csv"文件的前2列。务必加详细注释，提高代码可读性。**

In [7]:
# 每个类别是一篇文档；词频矩阵的行对应类别，列对应候选词。
types = list(category_texts)
corpus = [category_texts[news_type] for news_type in types]
vectorizer = CountVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False)
word_counts = vectorizer.fit_transform(corpus)

# 在 8 篇类别文档上计算 TF-IDF；权重越高，词越能代表该类别。
tfidf_matrix = TfidfTransformer().fit_transform(word_counts)
# 兼容 AI Studio 的旧版 sklearn：新旧接口返回相同顺序的词表。
if hasattr(vectorizer, "get_feature_names_out"):
    vocabulary = vectorizer.get_feature_names_out()
else:
    vocabulary = vectorizer.get_feature_names()

tfidf_keywords = []
for row in tfidf_matrix:
    weights = row.toarray().ravel()
    # 只在本类别出现过的词中取权重最高的 20 个。
    ranked = weights.argsort()[::-1]
    top_words = [vocabulary[i] for i in ranked if weights[i] > 0][:20]
    tfidf_keywords.append(" ".join(top_words))

# 先写入前两列。后续 cell 会在同一张表上添加第 3、4 列。
result = pd.DataFrame({"type": types, "tfidfkey": tfidf_keywords})
output_file = work_dir / "10780614.csv"
result.to_csv(output_file, index=False, encoding="utf-8")
print(result[["type", "tfidfkey"]].to_string(index=False))


type                                                               tfidfkey
  体育         比赛 火箭 姚明 球队 篮板 阿里 奇才 球员 易建联 赛季 防守 湖人 纳斯 助攻 进攻 内线 火箭队 体育讯 科比 阿联
  教育         签证 留学 学生 申请 美国 一个 学校 移民 大学 出国 申请人 留学生 中国 录取 英国 课程 签证官 专业 学习 工作
  股票   匿名 专家 warren 老师 曾文俊 网友 aries 谢谢 短期 持有 美股 王老师 请问 您好 压力 现在 今天 走势 支撑 坐堂
  游戏 电子竞技 游戏 海选 选手 比赛 玩家 总决赛 战队 第一组 魔兽争霸 PCGA 网游 WCG 赛区 星际争霸 电竞 中国 湖南 赛事 网吧
  科技      低俗 科学家 一个 投资 研究 支付宝 发现 进行 论文 互联网 航天飞机 中国 网站 人类 技术 海豚 信用卡 这种 表示 整治
  娱乐      电影 导演 票房 一个 影片 主持人 演员 上映 观众 听歌 blog 香港 万港元 电影节 梅兰芳 主演 执导 首映 李佳 这部
  彩票          赔率 主队 VS 主场 主胜 英超 足彩 客场 客胜 球队 比赛 联赛 切尔西 两队 胜负彩 平赔 客队 本场 球失 比值
  房产          一个 项目 户型 别墅 北京 中国 建筑 地产 乐居 现在 发展 市场 房地产 城市 生活 房子 一些 平米 投资 样板间


**使用TextRank提取news.csv中8个领域新闻的前20个关键词，将textrankkey保存到"work/学号.csv"文件的第3列。务必加详细注释，提高代码可读性。**

In [8]:
# TextRank 根据词语在文本窗口中的共现关系给词语排序。
# 依照 result 的类别顺序逐行提取，避免关键词和类别错位。
textrank_keywords = []
for news_type in result["type"]:
    words = jieba.analyse.textrank(category_texts[news_type], topK=20)
    textrank_keywords.append(" ".join(words))

result["textrankkey"] = textrank_keywords
result.to_csv(output_file, index=False, encoding="utf-8")
print(result[["type", "textrankkey"]].to_string(index=False))


type                                                    textrankkey
  体育   比赛 球队 火箭 防守 时间 进攻 球员 奇才 表现 篮板 问题 湖人 体育讯 对手 球迷 阿联 助攻 北京 训练 得分
  教育   学生 留学 美国 签证 申请 大学 中国 学校 英国 专业 工作 出国 需要 移民 选择 学习 留学生 教育 问题 国家
  股票    匿名 专家 网友 老师 经济 美国 持有 压力 问题 继续 公司 走势 请问 支撑 认为 政府 分析 黄金 市场 股票
  游戏 游戏 比赛 选手 电子竞技 玩家 中国 进行 总决赛 手机 网游 项目 赛事 战队 赛区 世界 报名 冠军 时间 参加 韩国
  科技  研究 进行 发现 投资 中国 网站 科学家 技术 公司 人类 认为 表示 互联网 美国 时间 提供 项目 网络 信息 报道
  娱乐  电影 导演 主持人 影片 票房 香港 观众 演员 觉得 中国 上映 拍摄 作品 知道 现场 电影节 市场 希望 故事 首映
  彩票    主队 赔率 主场 比赛 球队 公司 客场 足彩 本场 联赛 客胜 数据 彩票 看好 对手 奖金 客队 博彩 情况 推荐
  房产    项目 中国 北京 建筑 市场 发展 城市 生活 地产 问题 投资 空间 活动 设计 区域 户型 别墅 房子 国际 文化


**利用jieba提取news.csv中8个领域新闻的前20个关键词，将jiebakey保存到"work/学号.csv"文件的第4列。务必加详细注释，提高代码可读性。**

In [9]:
# jieba.analyse.extract_tags 是 jieba 自带的 TF-IDF 关键词提取接口。
# 虽然也使用 TF-IDF 思想，它与上面 sklearn 在 8 类文档上计算的结果可以不同。
jieba_keywords = []
for news_type in result["type"]:
    words = jieba.analyse.extract_tags(category_texts[news_type], topK=20)
    jieba_keywords.append(" ".join(words))

result["jiebakey"] = jieba_keywords
# 按题目要求固定四列顺序，一类新闻占一行。
result = result[["type", "tfidfkey", "textrankkey", "jiebakey"]]
result.to_csv(output_file, index=False, encoding="utf-8")
print(f"已保存 {len(result)} 个类别的关键词：{output_file}")
print(result.to_string(index=False))


已保存 8 个类别的关键词：work\10780614.csv
type                                                               tfidfkey                                                    textrankkey                                                               jiebakey
  体育         比赛 火箭 姚明 球队 篮板 阿里 奇才 球员 易建联 赛季 防守 湖人 纳斯 助攻 进攻 内线 火箭队 体育讯 科比 阿联   比赛 球队 火箭 防守 时间 进攻 球员 奇才 表现 篮板 问题 湖人 体育讯 对手 球迷 阿联 助攻 北京 训练 得分         比赛 球队 姚明 篮板 赛季 火箭 奇才 防守 易建联 体育讯 球员 助攻 阿里 湖人 纳斯 科比 进攻 马刺 新浪 火箭队
  教育         签证 留学 学生 申请 美国 一个 学校 移民 大学 出国 申请人 留学生 中国 录取 英国 课程 签证官 专业 学习 工作   学生 留学 美国 签证 申请 大学 中国 学校 英国 专业 工作 出国 需要 移民 选择 学习 留学生 教育 问题 国家         留学 签证 学生 申请 大学 美国 学校 出国 申请人 留学生 移民 一个 专业 英国 学习 签证官 录取 中国 选择 学费
  股票   匿名 专家 warren 老师 曾文俊 网友 aries 谢谢 短期 持有 美股 王老师 请问 您好 压力 现在 今天 走势 支撑 坐堂    匿名 专家 网友 老师 经济 美国 持有 压力 问题 继续 公司 走势 请问 支撑 认为 政府 分析 黄金 市场 股票   匿名 专家 warren 网友 曾文俊 老师 谢谢 aries 王老师 美股 请问 压力 持有 短期 您好 支撑 今天 A股 现在 走势
  游戏 电子竞技 游戏 海选 选手 比赛 玩家 总决赛 战队 第一组 魔兽争霸 PCGA 网游 WCG 赛区 星际争霸 电竞 中国 湖南 赛事 网吧 游戏 比赛 选手 电子竞技 玩家 中国 进行 总决赛 手机 网游 项目 赛事 战队 赛区 世界 报名 冠